---
title: "Glossary Terms"
format:
  html:
    code-fold: true
    code-summary: "Show code"
---

This notebook explores glossary terms in the ClimateKG graph by tracing how glossary entries are linked to report and concept structure.

We use the live ClimateKG Wikibase instance and the authenticated SPARQL endpoint.

**Query UI**: https://climatekg.tibwiki.io/query/

**SPARQL endpoint**: https://climatekg.tibwiki.io/query/proxy/sparql

This notebook follows the same query-first workflow as the earlier notebooks: validate the graph structure before interpreting the terminology and concept distribution.

In [1]:
import os
import pandas as pd
from SPARQLWrapper import JSON
from IPython.display import display, Markdown

from wikibase_auth import DEFAULT_SPARQL_ENDPOINT, DEFAULT_WIKIBASE_URL, build_sparql_client

os.environ.setdefault("CLIMATEKG_SPARQL_USERNAME", "ckg")
os.environ.setdefault("CLIMATEKG_SPARQL_PASSWORD", "fairdata")

SPARQL_ENDPOINT = os.getenv("CLIMATEKG_SPARQL_ENDPOINT", DEFAULT_SPARQL_ENDPOINT)
WIKIBASE_URL = os.getenv("CLIMATEKG_WIKIBASE_URL", DEFAULT_WIKIBASE_URL)

print(f"Endpoint: {SPARQL_ENDPOINT}")
print("Libraries imported successfully")

'ckg'

'fairdata'

Endpoint: https://climatekg.tibwiki.io/query/proxy/sparql
Libraries imported successfully


In [2]:
query = '''
PREFIX wd: <https://climatekg.tibwiki.io/entity/>
PREFIX wdt: <https://climatekg.tibwiki.io/prop/direct/>

SELECT ?term ?termLabel ?report ?reportLabel ?termType ?termTypeLabel
WHERE {
  ?term wdt:P1 wd:Q1 .
  ?term wdt:P3 ?report .
  OPTIONAL { ?term wdt:P1 ?termType . }
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
}
ORDER BY ?reportLabel ?termLabel
LIMIT 200
'''

sparql = build_sparql_client(SPARQL_ENDPOINT)
sparql.setQuery(query)
sparql.setReturnFormat(JSON)
results = sparql.query().convert()
rows = results.get("results", {}).get("bindings", [])
df = pd.DataFrame([{k: v["value"] for k, v in row.items()} for row in rows])

print(f"Rows returned: {len(df)}")
display(df.head(20))

Rows returned: 200


,term,report,termType,termLabel,reportLabel,termTypeLabel
0,https://climatekg.tibwiki.io/entity/Q1168,https://climatekg.tibwiki.io/entity/Q10,https://climatekg.tibwiki.io/entity/Q1,1.5°C warmer worlds,Special Report: Global Warming of 1.5°C,Category
1,https://climatekg.tibwiki.io/entity/Q1167,https://climatekg.tibwiki.io/entity/Q77,https://climatekg.tibwiki.io/entity/Q1,1.5°C pathway,Working Group I: Climate Change 2021 – The Phy...,Category
2,https://climatekg.tibwiki.io/entity/Q1169,https://climatekg.tibwiki.io/entity/Q77,https://climatekg.tibwiki.io/entity/Q1,13C,Working Group I: Climate Change 2021 – The Phy...,Category
3,https://climatekg.tibwiki.io/entity/Q1170,https://climatekg.tibwiki.io/entity/Q77,https://climatekg.tibwiki.io/entity/Q1,14C,Working Group I: Climate Change 2021 – The Phy...,Category
4,https://climatekg.tibwiki.io/entity/Q1172,https://climatekg.tibwiki.io/entity/Q77,https://climatekg.tibwiki.io/entity/Q1,Ablation,Working Group I: Climate Change 2021 – The Phy...,Category
5,https://climatekg.tibwiki.io/entity/Q1173,https://climatekg.tibwiki.io/entity/Q77,https://climatekg.tibwiki.io/entity/Q1,Abrupt change,Working Group I: Climate Change 2021 – The Phy...,Category
6,https://climatekg.tibwiki.io/entity/Q1174,https://climatekg.tibwiki.io/entity/Q77,https://climatekg.tibwiki.io/entity/Q1,Abrupt climate change,Working Group I: Climate Change 2021 – The Phy...,Category
7,https://climatekg.tibwiki.io/entity/Q259,https://climatekg.tibwiki.io/entity/Q77,https://climatekg.tibwiki.io/entity/Q1,Accumulation,Working Group I: Climate Change 2021 – The Phy...,Category
8,https://climatekg.tibwiki.io/entity/Q260,https://climatekg.tibwiki.io/entity/Q77,https://climatekg.tibwiki.io/entity/Q1,Active layer,Working Group I: Climate Change 2021 – The Phy...,Category
9,https://climatekg.tibwiki.io/entity/Q262,https://climatekg.tibwiki.io/entity/Q77,https://climatekg.tibwiki.io/entity/Q1,Adaptation,Working Group I: Climate Change 2021 – The Phy...,Category


## Interpretation

Each row connects a glossary term to a report or corpus item via the `P3` relationship. That lets us see the terminology in context rather than as isolated labels.

This is the right starting point for glossary analysis: once the relation is confirmed, the next step is to count terms by report, identify repeated concepts, and compare the terms against the glossary pages already generated in the site.